# Stage 12 — final visualization and residual failure audit

Read-only visualization of the final Stage 11 tracks. Stage 9 remains unchanged and continues to diagnose the pipeline through Stage 8.

In [ ]:
from importlib import import_module, reload

import numpy as np
import pandas as pd

from src.io import (
    PipelinePaths,
    load_npy_time_series,
    load_processed_dataset_inputs,
    load_stage11_outputs,
    open_sample,
)

stage12 = reload(import_module("src.12_final_visualization"))
napari_layers = reload(import_module("src.12_final_visualization.napari_layers"))
FinalVisualizationConfig = stage12.FinalVisualizationConfig
prepare_final_visualization_data = stage12.prepare_final_visualization_data
save_final_visualization_result = stage12.save_final_visualization_result
create_final_visualization_viewer = napari_layers.create_final_visualization_viewer

SAMPLE_ID = "44b6_0113de3b"
paths = PipelinePaths.discover()
config = FinalVisualizationConfig(
    boundary_margin_um=4.0,
    short_track_max_observations=2,
    show_boundary_tracks=False,
    show_modified_tracks=False,
)

In [ ]:
inputs = load_processed_dataset_inputs(SAMPLE_ID, paths=paths)
stage11 = load_stage11_outputs(paths=paths)
cells = pd.concat(
    [frame.assign(frame=index) for index, frame in enumerate(inputs.time_frames)],
    ignore_index=True,
)

raw = open_sample(paths.sample_zarr(SAMPLE_ID))
preprocessed, _ = load_npy_time_series(inputs.root / "preprocessing")
binary_mask, _ = load_npy_time_series(inputs.root / "masking")
instance_labels, _ = load_npy_time_series(inputs.root / "segmentation")
spatial_shape_zyx = tuple(int(value) for value in raw.shape[-3:])

visualization = prepare_final_visualization_data(
    stage11.tracks,
    cells,
    endpoint_classifications=stage11.endpoint_classifications,
    continuation_decisions=stage11.continuation_decisions,
    track_id_remap=stage11.track_id_remap,
    unresolved_endings=stage11.unresolved_endings,
    division_events=stage11.division_events,
    lineage_edges=stage11.lineage_edges,
    segmentation_events=stage11.segmentation_events,
    spatial_shape_zyx=spatial_shape_zyx,
    sequence_first_frame=0,
    sequence_last_frame=int(raw.shape[0]) - 1,
    config=config,
)

In [ ]:
for key, value in visualization.summary.items():
    print(f"{key}: {value}")

display(
    visualization.track_summary.loc[
        visualization.track_summary["failure_score"] > 0
    ].head(50)
)
display(visualization.failure_events.head(50))

In [ ]:
stage12_output = paths.processed_root / "stage_12_final_visualization"
save_final_visualization_result(visualization, stage12_output)
print("Stage 12 audit tables saved to:", stage12_output)


In [ ]:
viewer, failure_navigator = create_final_visualization_viewer(
    visualization,
    raw_volume=raw,
    preprocessed_volume=preprocessed,
    binary_mask_volume=binary_mask,
    instance_labels_volume=instance_labels,
    config=config,
)

In [ ]:
from diagnostics.cell_volume_extraction import add_cell_volume_extractor
from diagnostics.tracking_scene_extraction import add_tracking_scene_extractor

scene_extractor = add_tracking_scene_extractor(
    viewer=viewer,
    cells=cells,
    instance_labels_volume=instance_labels,
    binary_mask_volume=binary_mask,
    image_volumes={"raw": raw, "preprocessed": preprocessed},
    sample_id=SAMPLE_ID,
    save_root=paths.tracking_scenes,
    voxel_size_zyx=config.voxel_size_zyx_um,
    default_padding_zyx=config.scene_padding_zyx,
    cell_id_column="cell_id",
    frame_column="frame",
    source_metadata={
        "processed_dir": inputs.root,
        "cells_dir": inputs.root / "cells",
        "tracks_csv": paths.stage11_reconciliation / "tracks.csv",
        "source_zarr_array": paths.sample_zarr_array(SAMPLE_ID),
        "visualization_stage": "12_final_visualization",
    },
)
cell_extractor = add_cell_volume_extractor(
    viewer=viewer,
    cells=cells,
    image_volume=raw,
    sample_id=SAMPLE_ID,
    preprocessed_volume=preprocessed,
    binary_mask_volume=binary_mask,
    instance_labels_volume=instance_labels,
    voxel_size_zyx=config.voxel_size_zyx_um,
)

In [ ]:
import napari
napari.run()